In [4]:
import os
import numpy as np
import cv2
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

# --- Config ---
DATA_DIR  = "../train"   # a adapter vers ton dossier d'affiches
IMG_SIZE  = 128
VAL_SPLIT = 0.2
SEED      = 42
IMG_EXTS  = (".jpg")

np.random.seed(SEED)
print("Config ok")

Config ok


In [7]:
!pip install pandas


  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)


In [10]:
import pandas as pd

CSV_PATH   = "../train_labels.csv"   # a adapter
IMG_DIR    = "../train"             # dossier ou sont toutes les images

 
df = pd.read_csv(CSV_PATH, header=None, names=["filename", "label"])
print("Colonnes:", list(df.columns))
print(df.head())

# On adapte ces deux noms aux colonnes reelles affichees juste au-dessus
FN_COL, LB_COL = df.columns[0], df.columns[1]

paths  = [os.path.join(IMG_DIR, fn) for fn in df[FN_COL].astype(str)]
labels = df[LB_COL].values

classes = sorted(df[LB_COL].unique())
cls_to_idx = {c: i for i, c in enumerate(classes)}
y = np.array([cls_to_idx[l] for l in labels])

# Verifie que les fichiers existent vraiment
missing = [p for p in paths[:50] if not os.path.exists(p)]
print(f"\n{len(paths)} images, {len(classes)} classes: {classes}")
print("Exemples manquants (sur les 50 premiers):", missing[:5] if missing else "aucun")

# Distribution par classe (cle pour le F1 macro)
import collections
for c, n in sorted(collections.Counter(labels).items()):
    print(f"  classe {c}: {n}")

Colonnes: ['filename', 'label']
     filename  label
0  183840.jpg      0
1  590795.jpg      0
2  939930.jpg      2
3  769952.jpg      3
4  478458.jpg      0

3751 images, 5 classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Exemples manquants (sur les 50 premiers): aucun
  classe 0: 805
  classe 1: 161
  classe 2: 788
  classe 3: 837
  classe 4: 1160


In [11]:
def hsv_histogram(img_bgr, bins=(8, 8, 8)):
    """Histogramme HSV normalise. Capte la palette de couleurs de l'affiche."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins,
                        [0, 180, 0, 256, 0, 256])
    return cv2.normalize(hist, hist).flatten()

def hog_features(img_bgr):
    """HOG sur niveaux de gris. Capte structure et composition."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return hog(gray, orientations=9, pixels_per_cell=(16, 16),
               cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)

def extract_features(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Image illisible: {path}")
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return np.concatenate([hsv_histogram(img), hog_features(img)])

# Test sur une seule image avant de lancer l'extraction complete
sample = extract_features(paths[0])
print(f"Vecteur de features: dimension {sample.shape[0]}")

Vecteur de features: dimension 2276


In [14]:
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 KB 426.6 kB/s eta 0:00:00a 0:00:01


In [16]:
from tqdm import tqdm

# Extraction sur tout le dataset (peut prendre 1 a 3 min)
X = np.vstack([extract_features(p) for p in tqdm(paths)]).astype(np.float32)
print("Matrice X:", X.shape)

# Split stratifie: garde la meme proportion de chaque classe en train et val
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED
)
print("Train:", X_tr.shape, "| Val:", X_val.shape)

  0%|          | 0/3751 [00:00<?, ?it/s]

100%|██████████| 3751/3751 [00:29<00:00, 127.98it/s]


Matrice X: (3751, 2276)
Train: (3000, 2276) | Val: (751, 2276)


In [17]:
# Poids de classe: compense le desequilibre (classe 1 pesera ~7x plus que la classe 4)
cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
sample_weight = cw[y_tr]
print("Poids par classe:", {c: round(w, 2) for c, w in zip(np.unique(y_tr), cw)})

Poids par classe: {np.int64(0): np.float64(0.93), np.int64(1): np.float64(4.65), np.int64(2): np.float64(0.95), np.int64(3): np.float64(0.9), np.int64(4): np.float64(0.65)}


In [18]:
clf = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(classes),
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
)
clf.fit(
    X_tr, y_tr,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)],
)
print("Entrainement termine, arbres utilises:", clf.best_iteration_)

/home/ludo/git/ML_Projet/Computer_vision_challenge/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.824446 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 560176
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 2273
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[69]	valid_0's multi_logloss: 1.01303
Entrainement termine, arbres utilises: 69


In [19]:
y_pred = clf.predict(X_val)

f1_macro = f1_score(y_val, y_pred, average="macro")
f1_micro = f1_score(y_val, y_pred, average="micro")
f1_weighted = f1_score(y_val, y_pred, average="weighted")

print(f"F1 macro    : {f1_macro:.4f}   <- le score qui compte a priori")
print(f"F1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}\n")

print(classification_report(y_val, y_pred, target_names=[str(c) for c in classes], digits=3))

F1 macro    : 0.4918   <- le score qui compte a priori
F1 micro    : 0.6032
F1 weighted : 0.5915

              precision    recall  f1-score   support

           0      0.658     0.596     0.625       161
           1      0.000     0.000     0.000        32
           2      0.605     0.677     0.639       158
           3      0.545     0.536     0.541       168
           4      0.623     0.690     0.654       232

    accuracy                          0.603       751
   macro avg      0.486     0.500     0.492       751
weighted avg      0.582     0.603     0.592       751



In [20]:
import joblib

joblib.dump(
    {"model": clf, "classes": classes, "cls_to_idx": cls_to_idx, "img_size": IMG_SIZE},
    "baseline_lgbm.joblib",
)
print("Modele sauvegarde dans baseline_lgbm.joblib")

Modele sauvegarde dans baseline_lgbm.joblib


In [23]:
TEST_CSV = "../test_unlabels.csv"
TEST_DIR = "../test"
SUBMISSION_PATH = "submission.csv"

# Liste des images a predire (CSV sans en-tete, une colonne)
test_df = pd.read_csv(TEST_CSV, header=None, names=["filename"])
test_files = test_df["filename"].astype(str).tolist()
print(f"{len(test_files)} images a predire")

# Extraction des features, meme pipeline que le train
X_test = []
for i, fn in enumerate(test_files):
    X_test.append(extract_features(os.path.join(TEST_DIR, fn)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(test_files)} traitees")
X_test = np.vstack(X_test).astype(np.float32)

# Prediction: indice interne -> label d'origine
pred_idx = clf.predict(X_test)
idx_to_cls = {i: c for c, i in cls_to_idx.items()}
pred_labels = [idx_to_cls[i] for i in pred_idx]

# CSV de soumission
submission = pd.DataFrame({"Id": test_files, "label": pred_labels})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSoumission ecrite: {SUBMISSION_PATH}")
print(submission.head())
print("\nRepartition des predictions:")
print(submission["label"].value_counts().sort_index())

1412 images a predire
  500/1412 traitees
  1000/1412 traitees

Soumission ecrite: submission.csv
           Id  label
0  755725.jpg      2
1  239260.jpg      4
2  357874.jpg      4
3  503277.jpg      4
4  192830.jpg      4

Repartition des predictions:
label
0    274
1      4
2    350
3    287
4    497
Name: count, dtype: int64
